# Integración de las tres asignaturas

Este notebook toma los parquet que genera cada notebook de limpieza y los unifica en `data/integrado/`.

Decisiones de esta etapa:
- La clave de estudiante es `usuario` (formato `ALU_xxxx`) y la de materia es `materia` (`algebra`, `algoritmos`, `probabilidad`).
- En calificaciones, cada instrumento recibe un nombre legible (`instrumento`) y un tipo (`tipo_evaluacion`).
- Quien no rindió un instrumento queda como nulo (NaN) en la tabla ancha de notas. No se imputa 0 en esta etapa.

## 1. Configuración y carga

In [1]:
import re
from pathlib import Path
import pandas as pd
from utils_limpieza import sin_tildes

DATA = Path("../data")
SALIDA = DATA / "integrado"
SALIDA.mkdir(exist_ok=True)

# Clave normalizada de materia -> (carpeta, sufijo de los parquet de esa materia)
MATERIAS = {
    "algebra": ("algebra", "algebra"),
    "algoritmos": ("algoritmosestructuras", "algoritmos"),
    "probabilidad": ("probabilidadestadistica", "probabilidadestadistica"),
}


def leer(fuente, materia):
    carpeta, sufijo = MATERIAS[materia]
    return pd.read_parquet(DATA / carpeta / f"{fuente}_{sufijo}.parquet").assign(materia=materia)


def integrar(fuente):
    """Concatena la fuente de las tres materias y avisa si las columnas no coinciden."""
    dfs = {m: leer(fuente, m) for m in MATERIAS}
    base = set(dfs["algebra"].columns)
    for m, d in dfs.items():
        dif = set(d.columns) ^ base
        if dif:
            print(f"  ⚠ {fuente}/{m}: columnas distintas respecto de algebra: {sorted(dif)}")
    df = pd.concat(dfs.values(), ignore_index=True)
    print(f"{fuente}: {len(df)} filas {({m: len(d) for m, d in dfs.items()})}")
    return df


df_calificaciones = integrar("calificaciones")   # se espera aviso por tipo_evaluacion (solo Álgebra lo trae)
df_participacion = integrar("participacion")
df_trazabilidad = integrar("trazabilidad")
df_cruce = integrar("cruce_matriculados")

  ⚠ calificaciones/algoritmos: columnas distintas respecto de algebra: ['tipo_evaluacion']
  ⚠ calificaciones/probabilidad: columnas distintas respecto de algebra: ['tipo_evaluacion']
calificaciones: 1793 filas {'algebra': 479, 'algoritmos': 419, 'probabilidad': 895}
participacion: 957 filas {'algebra': 319, 'algoritmos': 319, 'probabilidad': 319}
trazabilidad: 149458 filas {'algebra': 41793, 'algoritmos': 44372, 'probabilidad': 63293}
cruce_matriculados: 957 filas {'algebra': 319, 'algoritmos': 319, 'probabilidad': 319}


In [2]:
def clave_materia(nombre):
    n = sin_tildes(nombre)
    for clave in MATERIAS:
        if clave in n:
            return clave
    raise ValueError(f"Materia sin clave normalizada: {nombre}")


df_matriculados = pd.read_parquet(DATA / "matriculados" / "matriculados.parquet")
df_matriculados = df_matriculados.rename(columns={"materia": "materia_nombre"})
df_matriculados["materia"] = df_matriculados["materia_nombre"].map(clave_materia)
df_matriculados.groupby("materia").size()

materia
algebra         165
algoritmos      167
probabilidad    167
dtype: int64

## 2. Esquema unificado de calificaciones

Cada Excel de calificaciones trae en su encabezado el título del instrumento (fila "Calificación:...") y el aula. Se leen de ahí, se limpian los símbolos que a veces anteceden al título (como "&#128073;") y se asignan dos columnas:
- `instrumento`: el título limpio (ej. "Actividad Rápida 4").
- `tipo_evaluacion`: una categoría deducida del título (`parcial`, `recuperatorio`, `autoevaluacion`, `actividad_rapida` o `evaluacion_semanal`).

In [3]:
import html


def limpiar_titulo(texto):
    t = html.unescape(texto)                                           # "&#128073;" -> emoji
    t = re.sub(r"[\U0001F000-\U0001FFFF\u2600-\u27BF\uFE0F]", "", t)       # emojis en cualquier posición
    t = re.sub(r"^[^\w]+", "", t)                                      # símbolos o espacios al inicio
    return re.sub(r"\s+", " ", t).strip()


def leer_cabecera(ruta):
    # Devuelve (aula, título) a partir de las primeras filas del Excel de calificaciones
    filas = pd.read_excel(ruta, header=None, nrows=8, usecols=[0])[0].dropna().astype(str)
    aula = filas[filas.str.match(r"(?i)^nombre del aula\s*:")]
    titulo = filas[filas.str.match(r"(?i)^calificaci[oó]n\s*:")]
    if titulo.empty:
        raise ValueError(f"{ruta.name}: no se encontró la fila 'Calificación:' en el encabezado")
    aula = re.sub(r"(?i)^nombre del aula\s*:\s*", "", aula.iloc[0]) if len(aula) else ""
    return aula, limpiar_titulo(re.sub(r"(?i)^calificaci[oó]n\s*:", "", titulo.iloc[0]))


# (materia, id del archivo = hhmmss del nombre) -> título del instrumento
TITULOS = {}
for materia, (carpeta, _) in MATERIAS.items():
    for ruta in sorted((DATA / carpeta).glob("calificaciones*.xlsx")):
        aula, titulo = leer_cabecera(ruta)
        if materia not in sin_tildes(aula):
            print(f"  ⚠ {ruta.name}: el aula '{aula}' no parece corresponder a {materia}")
        TITULOS[(materia, re.search(r"_(\d{6})\.xlsx$", ruta.name).group(1))] = titulo

pd.Series(TITULOS, name="instrumento").rename_axis(["materia", "id_archivo"]).to_frame()

instrumento
materia      id_archivo                                       
algebra      114929                            AUTOEVALUACION1
             114941                            AUTOEVALUACION2
             114944                                 1° Parcial
algoritmos   115518                  Autoevaluación - Módulo 1
             115523                  Autoevaluación - Unidad 2
             115528                  Autoevaluación - Unidad 3
             115533                  Autoevaluación - Unidad 4
             115538                  Autoevaluación - Unidad 5
probabilidad 115041                                  PARCIAL 1
             115045                         Actividad Rápida 1
             115049                         Actividad Rápida 2
             115055      Recuperatorio: Evaluación Semanal Nº1
             115059                     Evaluación Semanal Nº1
             115102                         Actividad Rápida 4
             115105                         Actividad Rápida 3
             115127                         Actividad Rápida 5
             115130                     Evaluación Semanal Nº2

In [4]:
def clasificar_tipo(titulo):
    t = sin_tildes(titulo)
    if "recuperatorio" in t:        # antes que "evaluacion semanal": un recuperatorio puede nombrar a otra evaluación
        return "recuperatorio"
    if "parcial" in t:
        return "parcial"
    if "autoevaluacion" in t:
        return "autoevaluacion"
    if "actividad rapida" in t:
        return "actividad_rapida"
    if "evaluacion semanal" in t:
        return "evaluacion_semanal"
    return None


cal = df_calificaciones
cal["id_archivo"] = cal["archivo"].str.extract(r"_(\d{6})\.xlsx$")[0]
cal["instrumento"] = pd.Series([TITULOS.get(k) for k in zip(cal["materia"], cal["id_archivo"])],
                               index=cal.index, dtype="string")
if cal["instrumento"].isna().any():
    raise ValueError("Archivos de calificaciones sin título: "
                     f"{cal.loc[cal['instrumento'].isna(), ['materia', 'archivo']].drop_duplicates().values.tolist()}")

# Tipo según el título; si no se reconoce, el que calculó el notebook de la materia (solo Álgebra, por nota máxima)
tipo_titulo = cal["instrumento"].map(clasificar_tipo)
tipo_previo = cal["tipo_evaluacion"].astype("string")
discrepancias = cal.assign(tipo_titulo=tipo_titulo, tipo_previo=tipo_previo)
discrepancias = discrepancias[discrepancias["tipo_titulo"].notna() & discrepancias["tipo_previo"].notna()
                              & (discrepancias["tipo_titulo"] != discrepancias["tipo_previo"])]
print(f"Instrumentos cuyo tipo por título difiere del tipo por nota máxima: {discrepancias['instrumento'].nunique()}")
cal["tipo_evaluacion"] = tipo_titulo.fillna(tipo_previo)

sin_tipo = cal[cal["tipo_evaluacion"].isna()]
if len(sin_tipo):
    raise ValueError(f"Instrumentos sin tipo reconocido: {sin_tipo['instrumento'].unique().tolist()}")

cal["observaciones"] = cal["observaciones"].astype("string").str.strip().str.lower()
cal["porcentaje"] = cal["nota_obtenida"] / cal["nota_max"] * 100
df_calificaciones = cal[["materia", "instrumento", "tipo_evaluacion", "usuario", "apellido", "nombre",
                         "observaciones", "nota_obtenida", "nota_max", "porcentaje", "archivo", "id_archivo"]]

# Cantidad de instrumentos por materia y tipo (Álgebra debería tener 2 parciales, 5 autoevaluaciones y 1 recuperatorio)
df_calificaciones.groupby(["materia", "tipo_evaluacion"])["instrumento"].nunique().unstack(fill_value=0)

Instrumentos cuyo tipo por título difiere del tipo por nota máxima: 0


tipo_evaluacion,actividad_rapida,autoevaluacion,evaluacion_semanal,parcial,recuperatorio
materia,,,,,
algebra,0,2,0,1,0
algoritmos,0,5,0,0,0
probabilidad,5,0,2,1,1


**Verificación de consistencia entre observaciones y la nota:** se busca, por instrumento, algún "no aprobada" con porcentaje igual o mayor al de algún "aprobada". Si aparece alguno, el instrumento tiene registros inconsistentes (o un criterio de aprobación que no es solo la nota).

In [5]:
apr = df_calificaciones[df_calificaciones["observaciones"] == "aprobada"].groupby(["materia", "instrumento"])["porcentaje"].min()
nap = df_calificaciones[df_calificaciones["observaciones"] == "no aprobada"].groupby(["materia", "instrumento"])["porcentaje"].max()
solapados = pd.concat([apr.rename("min_aprobada"), nap.rename("max_no_aprobada")], axis=1).dropna()
solapados = solapados[solapados["max_no_aprobada"] >= solapados["min_aprobada"]]
print(f"Valores de observaciones: {df_calificaciones['observaciones'].value_counts(dropna=False).to_dict()}")
print(f"Instrumentos con 'no aprobada' >= 'aprobada': {len(solapados)}")
solapados

Valores de observaciones: {'aprobada': 1712, 'no aprobada': 81}
Instrumentos con 'no aprobada' >= 'aprobada': 0


,,min_aprobada,max_no_aprobada
materia,instrumento,,


## 3. Calificaciones faltantes: tabla ancha por materia

Los archivos de calificaciones solo traen filas de quienes rindieron. Se arma una fila por usuario y una columna por instrumento, con nulo (NaN) cuando no rindió.
La base de usuarios sale del cruce con matriculados (unión de todas las fuentes), por lo que incluye a quienes no rindieron nada.

In [6]:
base = df_cruce[["materia", "usuario", "matriculado", "activo_no_inscripto"]]
notas_anchas = {}

for materia in MATERIAS:
    cal_m = df_calificaciones[df_calificaciones["materia"] == materia]
    orden = cal_m.drop_duplicates("instrumento").sort_values("id_archivo")["instrumento"].tolist()
    # pivot falla si hubiera más de una nota por usuario e instrumento (ya se deduplicó en cada notebook)
    ancho = cal_m.pivot(index="usuario", columns="instrumento", values="nota_obtenida")[orden]

    ancho = base[base["materia"] == materia].set_index("usuario").drop(columns="materia").join(ancho, how="left")
    ancho["instrumentos_rendidos"] = ancho[orden].notna().sum(axis=1)
    ancho["instrumentos_totales"] = len(orden)
    notas_anchas[materia] = ancho.reset_index()

    print(f"{materia}: {len(ancho)} usuarios, {len(orden)} instrumentos")
    print("  Usuarios según instrumentos rendidos:", ancho["instrumentos_rendidos"].value_counts().sort_index().to_dict())

notas_anchas["probabilidad"].head()

algebra: 319 usuarios, 3 instrumentos
  Usuarios según instrumentos rendidos: {0: 123, 1: 33, 2: 43, 3: 120}
algoritmos: 319 usuarios, 5 instrumentos
  Usuarios según instrumentos rendidos: {0: 173, 1: 40, 2: 26, 3: 17, 4: 39, 5: 24}
probabilidad: 319 usuarios, 9 instrumentos
  Usuarios según instrumentos rendidos: {0: 125, 1: 27, 2: 27, 3: 17, 4: 15, 5: 12, 6: 44, 7: 38, 8: 13, 9: 1}


,usuario,matriculado,activo_no_inscripto,PARCIAL 1,Actividad Rápida 1,Actividad Rápida 2,Recuperatorio: Evaluación Semanal Nº1,Evaluación Semanal Nº1,Actividad Rápida 4,Actividad Rápida 3,Actividad Rápida 5,Evaluación Semanal Nº2,instrumentos_rendidos,instrumentos_totales
0,ALU_1001,True,False,65.0,100.0,100.0,NaN,100.00,100.0,100.0,NaN,NaN,6,9
1,ALU_1002,True,False,70.0,100.0,100.0,NaN,68.75,NaN,100.0,NaN,NaN,5,9
2,ALU_1003,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,9
3,ALU_1004,True,False,95.0,100.0,100.0,NaN,81.25,87.5,100.0,100.0,NaN,7,9
4,ALU_1005,False,True,65.0,100.0,100.0,NaN,93.75,87.5,100.0,100.0,NaN,7,9


## 4. Consistencia del identificador entre fuentes

Se compara, para cada `usuario`, el nombre que figura en calificaciones, participación, trazabilidad y matriculados (normalizado: sin tildes, sin mayúsculas ni puntuación y sin importar el orden de las palabras).
Un mismo `usuario` con nombres distintos es un registro inconsistente. También se listan nombres iguales con más de un `usuario` (posibles cuentas duplicadas u homónimos).

In [7]:
def nombre_normalizado(texto):
    t = re.sub(r"[^a-z\s]", " ", sin_tildes(texto))
    return " ".join(sorted(t.split()))


def tabla_nombres(df, fuente, nombre_completo):
    t = df[["materia", "usuario"]].copy()
    t["nombre_norm"] = nombre_completo.map(nombre_normalizado)
    t["fuente"] = fuente
    return t.drop_duplicates()


def completo(df):
    return df["apellido"].astype("string") + " " + df["nombre"].astype("string")


nombres = pd.concat([
    tabla_nombres(df_calificaciones, "calificaciones", completo(df_calificaciones)),
    tabla_nombres(df_participacion, "participacion", completo(df_participacion)),
    tabla_nombres(df_trazabilidad, "trazabilidad", completo(df_trazabilidad)),
    tabla_nombres(df_matriculados, "matriculados", df_matriculados["alumno"].astype("string")),
], ignore_index=True)

por_usuario = nombres.groupby("usuario")["nombre_norm"].nunique()
inconsistentes = por_usuario[por_usuario > 1].index
print(f"Usuarios únicos: {por_usuario.size}")
print(f"Usuarios con nombres distintos entre fuentes: {len(inconsistentes)}")

por_nombre = nombres.groupby("nombre_norm")["usuario"].nunique()
homonimos = por_nombre[por_nombre > 1].index
print(f"Nombres con más de un usuario: {len(homonimos)}")

nombres[nombres["usuario"].isin(inconsistentes)].sort_values(["usuario", "fuente"])

Usuarios únicos: 319
Usuarios con nombres distintos entre fuentes: 0
Nombres con más de un usuario: 18


,materia,usuario,nombre_norm,fuente


In [8]:
# Detalle de los nombres compartidos por más de un usuario (posibles homónimos o cuentas duplicadas)
nombres[nombres["nombre_norm"].isin(homonimos)].drop_duplicates(["usuario", "nombre_norm"]).sort_values(["nombre_norm", "usuario"])

,materia,usuario,nombre_norm,fuente
72,algebra,ALU_1129,acosta santiago,calificaciones
140,algebra,ALU_1249,acosta santiago,calificaciones
65,algebra,ALU_1118,alejandro moreno,calificaciones
86,algebra,ALU_1150,alejandro moreno,calificaciones
124,algebra,ALU_1220,alejandro ortiz,calificaciones
156,algebra,ALU_1301,alejandro ortiz,calificaciones
696,algebra,ALU_1161,alvarez victoria,participacion
106,algebra,ALU_1179,alvarez victoria,calificaciones
58,algebra,ALU_1108,cabrera eduardo,calificaciones
821,algebra,ALU_1286,cabrera eduardo,participacion


## 5. Resumen de la integración

In [9]:
resumen = pd.DataFrame({
    "usuarios_calificaciones": df_calificaciones.groupby("materia")["usuario"].nunique(),
    "usuarios_participacion": df_participacion.groupby("materia")["usuario"].nunique(),
    "usuarios_trazabilidad": df_trazabilidad.groupby("materia")["usuario"].nunique(),
    "matriculados": df_matriculados.groupby("materia")["usuario"].nunique(),
    "activos_no_inscriptos": df_cruce.groupby("materia")["activo_no_inscripto"].sum(),
    "activos_con_baja": df_cruce.groupby("materia")["activo_con_baja"].sum(),
    "inscriptos_sin_actividad": df_cruce.groupby("materia")["inscripto_sin_actividad"].sum(),
})
resumen

,usuarios_calificaciones,usuarios_participacion,usuarios_trazabilidad,matriculados,activos_no_inscriptos,activos_con_baja,inscriptos_sin_actividad
materia,,,,,,,
algebra,196,319,268,165,105,2,2
algoritmos,146,319,276,167,111,2,2
probabilidad,194,319,273,167,109,3,3


## 6. Resumen de archivos creados

In [11]:
display(df_calificaciones.head(3))

,materia,instrumento,tipo_evaluacion,usuario,apellido,nombre,observaciones,nota_obtenida,nota_max,porcentaje,archivo,id_archivo
0,algebra,AUTOEVALUACION1,autoevaluacion,ALU_1001,Gómez,Manuel,aprobada,9.0,10.0,90.0,calificaciones_alumnos_20251024_114929.xlsx,114929
1,algebra,AUTOEVALUACION1,autoevaluacion,ALU_1004,Flores,Gabriel,aprobada,9.0,10.0,90.0,calificaciones_alumnos_20251024_114929.xlsx,114929
2,algebra,AUTOEVALUACION1,autoevaluacion,ALU_1005,Vega,Agustín,aprobada,9.0,10.0,90.0,calificaciones_alumnos_20251024_114929.xlsx,114929


In [12]:
display(df_participacion.head(3))

,usuario,apellido,nombre,aula,perfil,fecha_alta,avance,interacciones,primera_interaccion,ultima_interaccion,...,foros_cantidad_de_intervenciones_del_usuario,anuncios_cantidad,anuncios_cargados,noticias_cantidad,noticias_cargados,tiempo_estimado_de_uso_min,fechas_inconsistentes,valores_negativos,avance_fuera_de_rango,materia
0,ALU_1001,Gómez,Manuel,TUCD-MD-Álgebra 2025,Alumnos,2025-08-05,86,1015,2025-08-05 10:50:31,2025-10-23 21:47:55,...,0,0,0,9,0,607.0,False,False,False,algebra
1,ALU_1002,Silva,Diego,TUCD-MD-Álgebra 2025,Alumnos,2025-08-05,90,616,2025-08-05 14:16:21,2025-10-23 19:44:57,...,0,0,0,9,0,389.0,False,False,False,algebra
2,ALU_1003,Castro,Federico,TUCD-MD-Álgebra 2025,Alumnos,2025-08-05,0,0,NaT,NaT,...,0,0,0,9,0,0.0,False,False,False,algebra


In [13]:
display(df_trazabilidad.head(3))

,fecha,ip_usuario,usuario,apellido,nombre,aula,perfil,seccion,accion,descripcion,materia
0,2025-08-05 10:50:31,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Álgebra 2025,Alumnos,Inicio,Acceder,Acceso a la sección 'Inicio',algebra
1,2025-08-11 13:29:48,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Álgebra 2025,Alumnos,Inicio,Acceder,Acceso a la sección 'Inicio',algebra
2,2025-08-11 13:29:57,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Álgebra 2025,Alumnos,Presentación,Acceder,Acceso a la sección 'Presentación',algebra


In [14]:
display(df_matriculados.head(3))

,archivo,materia_nombre,usuario,alumno,identificacion,comision,estado_insc,instancia,fecha_inscripcion,nro_de_transaccion,materia
0,Algoritmos y Estructuras de Datos.xlsx,Algoritmos y Estructuras de Datos,ALU_1248,"PÉREZ, FELIPE",ALU_1248,2320,Aceptada,Regularidad,2025-07-31 11:06:04,298521,algoritmos
1,Algoritmos y Estructuras de Datos.xlsx,Algoritmos y Estructuras de Datos,ALU_1212,"ORTIZ, GONZALO",ALU_1212,2320,Aceptada,Regularidad,2025-08-04 21:13:11,299608,algoritmos
2,Algoritmos y Estructuras de Datos.xlsx,Algoritmos y Estructuras de Datos,ALU_1089,"MORENO, BRUNO",ALU_1089,2320,Aceptada,Regularidad,2025-08-04 00:43:56,299233,algoritmos


In [15]:
display(df_cruce.head(3))

,usuario,matriculado,en_trazabilidad,en_participacion,en_calificaciones,estado_insc,instancia,activo_no_inscripto,activo_con_baja,inscripto_sin_actividad,materia
0,ALU_1001,True,True,True,True,Aceptada,Regularidad,False,False,False,algebra
1,ALU_1002,True,True,True,True,Aceptada,Regularidad,False,False,False,algebra
2,ALU_1003,False,False,True,False,<NA>,<NA>,False,False,False,algebra


In [16]:
display(resumen.head(3))

,usuarios_calificaciones,usuarios_participacion,usuarios_trazabilidad,matriculados,activos_no_inscriptos,activos_con_baja,inscriptos_sin_actividad
materia,,,,,,,
algebra,196,319,268,165,105,2,2
algoritmos,146,319,276,167,111,2,2
probabilidad,194,319,273,167,109,3,3


In [25]:
print("Algebra")
display(notas_anchas['algebra'].head(3))

print("Algoritmos")
display(notas_anchas['algoritmos'].head(3))

print("Probabilidad")
display(notas_anchas['probabilidad'].head(3))

Algebra


,usuario,matriculado,activo_no_inscripto,AUTOEVALUACION1,AUTOEVALUACION2,1° Parcial,instrumentos_rendidos,instrumentos_totales
0,ALU_1001,True,False,9.0,5.83,24.0,3,3
1,ALU_1002,True,False,NaN,10.00,17.0,2,3
2,ALU_1003,False,False,NaN,NaN,NaN,0,3


Algoritmos


,usuario,matriculado,activo_no_inscripto,Autoevaluación - Módulo 1,Autoevaluación - Unidad 2,Autoevaluación - Unidad 3,Autoevaluación - Unidad 4,Autoevaluación - Unidad 5,instrumentos_rendidos,instrumentos_totales
0,ALU_1001,True,False,NaN,NaN,NaN,NaN,NaN,0,5
1,ALU_1002,True,False,NaN,NaN,NaN,NaN,NaN,0,5
2,ALU_1003,False,True,NaN,NaN,NaN,NaN,NaN,0,5


Probabilidad


,usuario,matriculado,activo_no_inscripto,PARCIAL 1,Actividad Rápida 1,Actividad Rápida 2,Recuperatorio: Evaluación Semanal Nº1,Evaluación Semanal Nº1,Actividad Rápida 4,Actividad Rápida 3,Actividad Rápida 5,Evaluación Semanal Nº2,instrumentos_rendidos,instrumentos_totales
0,ALU_1001,True,False,65.0,100.0,100.0,NaN,100.00,100.0,100.0,NaN,NaN,6,9
1,ALU_1002,True,False,70.0,100.0,100.0,NaN,68.75,NaN,100.0,NaN,NaN,5,9
2,ALU_1003,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,9


## 7. Exportación

In [10]:
salidas = {
    "calificaciones_integrado": df_calificaciones,
    "participacion_integrado": df_participacion,
    "trazabilidad_integrado": df_trazabilidad,
    "matriculados_integrado": df_matriculados,
    "cruce_matriculados_integrado": df_cruce,
    "resumen_integracion": resumen.reset_index(),
    **{f"notas_anchas_{m}": d for m, d in notas_anchas.items()},
}
for nombre, d in salidas.items():
    d.to_parquet(SALIDA / f"{nombre}.parquet", index=False)
    print(f"{nombre}: {len(d)} filas, {d.shape[1]} columnas")

calificaciones_integrado: 1793 filas, 12 columnas
participacion_integrado: 957 filas, 33 columnas
trazabilidad_integrado: 149458 filas, 11 columnas
matriculados_integrado: 499 filas, 11 columnas
cruce_matriculados_integrado: 957 filas, 11 columnas
resumen_integracion: 3 filas, 8 columnas
notas_anchas_algebra: 319 filas, 8 columnas
notas_anchas_algoritmos: 319 filas, 10 columnas
notas_anchas_probabilidad: 319 filas, 14 columnas
